In [1]:
import torch
import itertools
import os
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

In [2]:
result_dir = Path("/data/scratch/pduan/gcvi_05-28_output/")

In [3]:
elbo_not_converge_cases = []
favi_not_converge_cases = []
elbo_invalid_k_hat_cases = 0
favi_invalid_k_hat_cases = 0
result_files = os.listdir(result_dir)
for i, item in enumerate(result_files):
    full_path = result_dir / item
    results = torch.load(full_path, map_location="cpu")
    elbo_not_converge = 0
    favi_not_converge = 0
    elbo_invalid_k_hat = 0
    favi_invalid_k_hat = 0
    elbo_error = None
    favi_error = None
    task_name = None
    results_num = len(results)
    for r in results:
        task_name = r["task"]
        if r["elbo_k_hat"] is None or \
            (torch.isinf(r["elbo_k_hat"]).any() | torch.isnan(r["elbo_k_hat"]).any()):
            elbo_invalid_k_hat += 1
        if r["favi_k_hat"] is None or \
            (torch.isinf(r["favi_k_hat"]).any() | torch.isnan(r["favi_k_hat"]).any()):
            favi_invalid_k_hat += 1

        if r["elbo_test_dict_list"] is None:
            elbo_not_converge += 1
        if r["favi_test_dict_list"] is None:
            favi_not_converge += 1
        elbo_error = r["elbo_error"]
        favi_error = r["favi_error"]
    if elbo_not_converge > 0:
        elbo_not_converge_cases.append(task_name)
    if elbo_invalid_k_hat > 0:
        elbo_invalid_k_hat_cases += 1
    if favi_not_converge > 0:
        favi_not_converge_cases.append(task_name)
    if favi_invalid_k_hat > 0:
        favi_invalid_k_hat_cases += 1 
    print("=" * 50)
    print(f"[{i + 1}] task name: {task_name}")
    print(f"elbo can't converge: {elbo_not_converge} / {results_num}")
    print(f"elbo invalid k hat: {elbo_invalid_k_hat} / {results_num}")
    print(f"favi can't converge: {favi_not_converge} / {results_num}")
    print(f"favi invalid k hat: {favi_invalid_k_hat} / {results_num}")
    # if elbo_error is None:
    #     print("elbo error: None")
    # else:
    #     print(f"elbo error: {elbo_error[:20]}")
    # if favi_error is None:
    #     print("favi error: None")
    # else:
    #     print(f"favi error: {favi_error[:20]}")
print("=" * 100)
print("Summary:")
print(f"elbo not converge cases: {len(elbo_not_converge_cases)} / {len(result_files)} ({len(elbo_not_converge_cases) / len(result_files):.4f})")
print(f"elbo not converge cases: {elbo_not_converge_cases}")
print(f"elbo invalid k hat cases: {elbo_invalid_k_hat_cases} / {len(result_files)} ({elbo_invalid_k_hat_cases / len(result_files):.4f})")
print(f"favi not converge cases: {len(favi_not_converge_cases)} / {len(result_files)} ({len(favi_not_converge_cases) / len(result_files):.4f})")
print(f"favi not converge cases: {favi_not_converge_cases}")
print(f"favi invalid k hat cases: {favi_invalid_k_hat_cases} / {len(result_files)} ({favi_invalid_k_hat_cases / len(result_files):.4f})")

[1] task name: arm_log10earn_height
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36
favi invalid k hat: 36 / 36
[2] task name: arm_electric_1b
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36
favi invalid k hat: 36 / 36
[3] task name: arm_logearn_height_male
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36
favi invalid k hat: 36 / 36
[4] task name: arm_electric_1a
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36
favi invalid k hat: 36 / 36
[5] task name: arm_kidiq_interaction_z
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36
favi invalid k hat: 36 / 36
[6] task name: arm_electric_1b_chr
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36
favi invalid k hat: 36 / 36
[7] task name: arm_kidscore_momhs
elbo can't converge: 0 / 36
elbo invalid k hat: 36 / 36
favi can't converge: 0 / 36

In [4]:
def get_est_mu_sigma2(results, tag):
    est_mu = []
    for r in results:
        tmp = []
        for d in r[f"{tag}_test_dict_list"]:
            tmp.append(d["est_mu"])
        est_mu.append(torch.stack(tmp, dim=0))  # (num_obs, k)
    est_mu = torch.stack(est_mu, dim=0)  # (r, num_obs, k)
    est_sigma2 = []
    for r in results:
        tmp = []
        for d in r[f"{tag}_test_dict_list"]:
            tmp.append(d["est_sigma2"])
        est_sigma2.append(torch.stack(tmp, dim=0))  # (num_obs, k)
    est_sigma2 = torch.stack(est_sigma2, dim=0)  # (r, num_obs, k)
    return est_mu, est_sigma2

In [5]:
def kl_div_two_normal(p_mu, p_sigma2, q_mu, q_sigma2):
    return torch.log(q_sigma2.sqrt()) - torch.log(p_sigma2.sqrt()) + (p_sigma2 + (p_mu - q_mu) ** 2) / (2 * q_sigma2) - 0.5

In [6]:
def get_kl_for_repeats(est_mu_sigma2: torch.Tensor):
    # est_mu_sigma2: (r, num_obs, k, 2)
    assert est_mu_sigma2.shape[-1] == 2
    combs = torch.tensor(list(itertools.combinations(range(est_mu_sigma2.shape[0]), 2)))  # (c, 2)
    perms = torch.cat([combs, combs.flip(dims=[-1])], dim=0)  # (2c, 2)
    perm_est_mu_sigma2 = est_mu_sigma2[perms]  # (2c, 2, num_obs, k, 2)
    return kl_div_two_normal(p_mu=perm_est_mu_sigma2[:, 0, :, :, 0],
                            p_sigma2=perm_est_mu_sigma2[:, 0, :, :, 1],
                            q_mu=perm_est_mu_sigma2[:, 1, :, :, 0],
                            q_sigma2=perm_est_mu_sigma2[:, 1, :, :, 1])  # (2c, num_obs, k)

In [7]:
def wasserstein_distance_to_unif(u: torch.Tensor):
    assert u.ndim == 3  # (r, k, s)
    unif_samples = torch.linspace(0.0, 1.0, u.shape[-1]).view(1, 1, -1)
    sorted_u = torch.sort(u, dim=-1, descending=False)[0]  # (r, k, s)
    return torch.abs(sorted_u - unif_samples).mean(dim=-1)  # (r, k)

In [8]:
def extract_vsbc(results, tag):
    vsbc_list = []
    for r in results:
        tmp = []
        for v in r[f"{tag}_vsbc"]:
            tmp.append(v)
        vsbc_list.append(torch.stack(tmp, dim=0))  # (k, s)
    return torch.stack(vsbc_list, dim=0)  # (r, k, s)

In [ ]:
result_files = os.listdir(result_dir)
elbo_avg_kl_r = []
elbo_std_mu = []
elbo_std_sigma2 = []
elbo_vsbc_to_unif_dist = []
favi_avg_kl_r = []
favi_std_mu = []
favi_std_sigma2 = []
favi_vsbc_to_unif_dist = []
favi_elbo_kl_ratio = []
favi_elbo_std_mu_ratio = []
favi_elbo_std_sigma2_ratio = []
favi_elbo_vsbc_d_ratio = []
task_name_list = []
for i, item in enumerate(result_files):
    full_path = result_dir / item
    results = torch.load(full_path, map_location="cpu")
    task_name = results[0]["task"]
    task_name_list.append(task_name)
    find_elbo_error = any([r["elbo_error"] is not None for r in results])
    find_favi_error = any([r["favi_error"] is not None for r in results])
    print("=" * 50)
    print(f"[{i + 1}] task name: {task_name}")
    print(f"elbo error: {find_favi_error}")
    print(f"favi error: {find_elbo_error}")

    if not find_elbo_error:
        elbo_est_mu, elbo_est_sigma2 = get_est_mu_sigma2(results, tag="elbo")
        assert (elbo_est_sigma2 > 0).all()
        print(f"std(elbo_est_mu): {elbo_est_mu.std(dim=0).mean().item():.3e}")
        elbo_std_mu.append(elbo_est_mu.std(dim=0).mean().item())
        print(f"std(elbo_est_sigma2): {elbo_est_sigma2.std(dim=0).mean().item():.3e}")
        elbo_std_sigma2.append(elbo_est_sigma2.std(dim=0).mean().item())
        elbo_kl_r = get_kl_for_repeats(torch.stack([elbo_est_mu, elbo_est_sigma2], dim=-1))
        print(f"elbo avg kl for repeats: {elbo_kl_r.mean().item():.3e}")
        elbo_avg_kl_r.append(elbo_kl_r.mean().item())
        elbo_vsbc = extract_vsbc(results, tag="elbo")  # (r, k, s)
        elbo_vsbc_d = wasserstein_distance_to_unif(elbo_vsbc)  # (r, k)
        print(f"elbo vsbc wasserstein dist to unif: {elbo_vsbc_d.mean().item():.3e}")
        elbo_vsbc_to_unif_dist.append(elbo_vsbc_d.mean().item())
    else:
        print("skip elbo, due to elbo error")
        elbo_std_mu.append(torch.nan)
        elbo_std_sigma2.append(torch.nan)
        elbo_avg_kl_r.append(torch.nan)
        elbo_vsbc_to_unif_dist.append(torch.nan)
    
    if not find_favi_error:
        favi_est_mu, favi_est_sigma2 = get_est_mu_sigma2(results, tag="favi")
        assert (favi_est_sigma2 > 0).all()
        print(f"std(favi_est_mu): {favi_est_mu.std(dim=0).mean().item():.3e}")
        favi_std_mu.append(favi_est_mu.std(dim=0).mean().item())
        print(f"std(favi_est_sigma2): {favi_est_sigma2.std(dim=0).mean().item():.3e}")
        favi_std_sigma2.append(favi_est_sigma2.std(dim=0).mean().item())
        favi_kl_r = get_kl_for_repeats(torch.stack([favi_est_mu, favi_est_sigma2], dim=-1))
        print(f"favi avg kl for repeats: {favi_kl_r.mean().item():.3e}")
        favi_avg_kl_r.append(favi_kl_r.mean().item())
        favi_vsbc = extract_vsbc(results, tag="favi")  # (r, k, s)
        favi_vsbc_d = wasserstein_distance_to_unif(favi_vsbc)  # (r, k)
        print(f"favi vsbc wasserstein dist to unif: {favi_vsbc_d.mean().item():.3e}")
        favi_vsbc_to_unif_dist.append(favi_vsbc_d.mean().item())
        if not find_elbo_error:
            if elbo_est_mu.std(dim=0).mean().item() > 0:
                favi_elbo_std_mu_ratio.append(favi_est_mu.std(dim=0).mean().item() / elbo_est_mu.std(dim=0).mean().item())
            if elbo_est_sigma2.std(dim=0).mean().item() > 0:
                favi_elbo_std_sigma2_ratio.append(favi_est_sigma2.std(dim=0).mean().item() / elbo_est_sigma2.std(dim=0).mean().item())
            favi_elbo_kl_ratio.append(favi_kl_r.mean().item() / elbo_kl_r.mean().item())
            favi_elbo_vsbc_d_ratio.append(favi_vsbc_d.mean().item() / elbo_vsbc_d.mean().item())
    else:
        print("skip favi, due to favi error")
        favi_std_mu.append(torch.nan)
        favi_std_sigma2.append(torch.nan)
        favi_avg_kl_r.append(torch.nan)
        favi_vsbc_to_unif_dist.append(torch.nan)

print()
print("+" * 100)
print("Summary:")
elbo_avg_kl_r = torch.tensor(elbo_avg_kl_r)
assert not torch.isinf(elbo_avg_kl_r).any()
elbo_std_mu = torch.tensor(elbo_std_mu)
assert not torch.isinf(elbo_std_mu).any()
elbo_std_sigma2 = torch.tensor(elbo_std_sigma2)
assert not torch.isinf(elbo_std_sigma2).any()
elbo_vsbc_to_unif_dist = torch.tensor(elbo_vsbc_to_unif_dist)
assert not torch.isinf(elbo_vsbc_to_unif_dist).any()

favi_avg_kl_r = torch.tensor(favi_avg_kl_r)
assert not torch.isinf(favi_avg_kl_r).any()
favi_std_mu = torch.tensor(favi_std_mu)
assert not torch.isinf(favi_std_mu).any()
favi_std_sigma2 = torch.tensor(favi_std_sigma2)
assert not torch.isinf(favi_std_sigma2).any()
favi_vsbc_to_unif_dist = torch.tensor(favi_vsbc_to_unif_dist)
assert not torch.isinf(favi_vsbc_to_unif_dist).any()

check_less_than = lambda a, b: ((a < b).sum() - torch.isnan(a).sum() + torch.isnan(b).sum()).item()
result_files_len = len(result_files)
def print_less_than_info(favi_d, elbo_d, tag):
    print(f"favi {tag} < elbo {tag}: "\
            f"{check_less_than(favi_d, elbo_d)}/{result_files_len} "\
            f"({check_less_than(favi_d, elbo_d) / result_files_len:.3f})")
    not_less_than = ~(favi_d < elbo_d)
    not_less_than_cases = [t for i, t in enumerate(task_name_list) if not_less_than[i]]
    print(f"favi {tag} >= elbo {tag} cases: {not_less_than_cases}")
    favi_nan = torch.isnan(favi_d)
    favi_nan_cases = [t for i, t in enumerate(task_name_list) if favi_nan[i]]
    print(f"favi {tag} nan cases: {favi_nan_cases}")
    elbo_nan = torch.isnan(elbo_d)
    elbo_nan_cases = [t for i, t in enumerate(task_name_list) if elbo_nan[i]]
    print(f"elbo {tag} nan cases: {elbo_nan_cases}")
    return [result_files[i] 
            for i in range(len(task_name_list)) 
            if not_less_than[i] and (not favi_nan[i]) and (not elbo_nan[i])]
std_mu_not_less_than_cases = print_less_than_info(favi_std_mu, elbo_std_mu, tag="std(mu)")
std_sigma2_not_less_than_cases = print_less_than_info(favi_std_sigma2, elbo_std_sigma2, tag="std(sigma2)")
kl_not_less_than_cases = print_less_than_info(favi_avg_kl_r, elbo_avg_kl_r, tag="kl")
vsbc_d_not_less_than_cases = print_less_than_info(favi_vsbc_to_unif_dist, elbo_vsbc_to_unif_dist, tag="vsbc to unif dist")

favi_elbo_std_mu_ratio = torch.tensor(favi_elbo_std_mu_ratio)
favi_elbo_std_sigma2_ratio = torch.tensor(favi_elbo_std_sigma2_ratio)
favi_elbo_kl_ratio = torch.tensor(favi_elbo_kl_ratio)
favi_elbo_vsbc_d_ratio = torch.tensor(favi_elbo_vsbc_d_ratio)
# print(f"mean of std(mu) ratio: {favi_elbo_std_mu_ratio.mean().item():.3e}")
print(f"median of std(mu) ratio: {favi_elbo_std_mu_ratio.median().item():.3e}")
# print(f"mean of std(sigma2) ratio: {favi_elbo_std_sigma2_ratio.mean().item():.3e}")
print(f"median of std(sigma2) ratio: {favi_elbo_std_sigma2_ratio.median().item():.3e}")
# print(f"mean of kl ratio: {favi_elbo_kl_ratio.mean().item():.3e}")
print(f"median of kl ratio: {favi_elbo_kl_ratio.median().item():.3e}")
# print(f"mean of vsbc d ratio: {favi_elbo_vsbc_d_ratio.mean().item():.3e}")
print(f"median of vsbc d ratio: {favi_elbo_vsbc_d_ratio.median().item():.3e}")
print("+" * 100)

[1] task name: arm_log10earn_height
elbo error: False
favi error: False
std(elbo_est_mu): 7.779e+01
std(elbo_est_sigma2): 9.442e-01
elbo avg kl for repeats: 3.604e+03
elbo vsbc wasserstein dist to unif: 2.386e-01
std(favi_est_mu): 1.855e-01
std(favi_est_sigma2): 3.356e-01
favi avg kl for repeats: 6.818e-02
favi vsbc wasserstein dist to unif: 1.705e-01
[2] task name: arm_electric_1b
elbo error: False
favi error: False
std(elbo_est_mu): 7.294e+02
std(elbo_est_sigma2): 1.259e+00
elbo avg kl for repeats: 5.290e+07
elbo vsbc wasserstein dist to unif: 2.625e-01
std(favi_est_mu): 3.944e+00
std(favi_est_sigma2): 1.495e-02
favi avg kl for repeats: 5.511e+00
favi vsbc wasserstein dist to unif: 2.219e-01
[3] task name: arm_logearn_height_male
elbo error: False
favi error: False
std(elbo_est_mu): 1.108e+02
std(elbo_est_sigma2): 1.311e+00
elbo avg kl for repeats: 1.672e+05
elbo vsbc wasserstein dist to unif: 2.400e-01
std(favi_est_mu): 1.488e-01
std(favi_est_sigma2): 3.887e-03
favi avg kl for repea

In [ ]:
def load_data(data_path, load_favi):
    data = torch.load(data_path, map_location=torch.device("cpu"))
    model_num = len(data)
    task_name = data[0]["task"]
    obs_num = len(data[0]["favi_test_dict_list"])
    theta_num = data[0]["favi_test_dict_list"][0]["est_mu"].shape[0]

    est_sigma2 = torch.zeros(model_num, obs_num, theta_num)
    est_mu = torch.zeros(model_num, obs_num, theta_num)

    for i in range(model_num):
        if load_favi:
            test_dict_list = data[i]["favi_test_dict_list"]
        else:
            test_dict_list = data[i]["elbo_test_dict_list"]
        for obs_index in range(obs_num):
            est_mu[i, obs_index] = test_dict_list[obs_index]["est_mu"]
            est_sigma2[i, obs_index] = test_dict_list[obs_index]["est_sigma2"]
    return {
        "task_name": task_name,
        "obs_num": obs_num,
        "theta_num": theta_num,
        "est_mu": est_mu,
        "est_sigma2": est_sigma2,
    }

In [ ]:
def plot_mu(data_path):
    if not os.path.isfile(data_path):
        print(f"skip {data_path}")
        return None
    
    favi_data_dict = load_data(data_path, load_favi=True)
    obs_num = favi_data_dict["obs_num"]
    theta_num = favi_data_dict["theta_num"]
    favi_est_mu = favi_data_dict["est_mu"]
    task_name: str = favi_data_dict["task_name"]

    elbo_data_dict = load_data(data_path, load_favi=False)
    elbo_est_mu = elbo_data_dict["est_mu"]

    fig, axes = plt.subplots(theta_num, obs_num, 
                             figsize=(obs_num * 2.5, theta_num * 2.5),
                             constrained_layout=True)
    
    axes: list[plt.Axes] = axes.flatten()
    for i, ax in enumerate(axes):
        ri, ci = i % obs_num, i // obs_num
        # ax.boxplot([est_mu[:, ri, ci], 
        #             elbo_est_mu[:, ri, ci]],
        #             labels=["FAVI", "ELBO"])
        sns.kdeplot(favi_est_mu[:, ri, ci], fill=True, 
                    label="FAVI", color="blue", alpha=0.3,
                    ax=ax)
        sns.kdeplot(elbo_est_mu[:, ri, ci], fill=True,
                    label="ELBO", color="red", alpha=0.3,
                    ax=ax)
        ax.set_title(f"obs {ri}; theta {ci}")
    
    # fig.tight_layout()
    # fig.subplots_adjust(right=0.8, top=theta_num / (0.5 + theta_num))
    fig.suptitle(task_name.replace("_", " ").upper(), 
                 fontsize=12)
    # fig.legend(["FAVI", "ELBO"], 
    #            loc="center right", ncol=1, fontsize=12,
    #            bbox_to_anchor=(0.85, 0.5))
    fig.show()

In [ ]:
for kl_nc in kl_not_less_than_cases:
    plot_mu(result_dir / kl_nc)

In [ ]:
# for vsbc_nc in vsbc_d_not_less_than_cases:
#     plot_mu(result_dir / vsbc_nc)

In [ ]:
def plot_vsbc(favi_vsbc_matrix: torch.Tensor, elbo_vsbc_matrix: torch.Tensor, suptitle):
    theta_dim = favi_vsbc_matrix.shape[0]
    nrows = theta_dim // 4 + 1 if theta_dim % 4 != 0 else theta_dim // 4
    ncols = 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = axes.flatten()
    for i, (ax, favi_vsbc_samples, elbo_vsbc_samples) in enumerate(zip(axes, favi_vsbc_matrix, elbo_vsbc_matrix)):
        sns.kdeplot(favi_vsbc_samples, 
                    ax=ax, 
                    clip=(0, 1),
                    fill=True,
                    bw_adjust=0.7,
                    color="green",
                    alpha=0.3,
                    label="FAVI")
        sns.kdeplot(elbo_vsbc_samples, 
                    ax=ax, 
                    clip=(0, 1),
                    fill=True,
                    bw_adjust=0.7,
                    color="blue",
                    alpha=0.3,
                    label="ELBO")
        if i == 0:
            ax.legend()
        ax.axvline(x=0.5, color="red", linestyle="--")
        ax.set_title(f"theta[{i}]")
    if nrows > 1:
        for ax in axes[theta_dim:]:
            ax.remove()
    fig.suptitle(suptitle)
    fig.tight_layout()
    fig.show()

In [ ]:
def plot_vsbc_for_case(data_path):
    r = torch.load(data_path, map_location="cpu")[0]
    task_name = r["task"].replace("_", " ").upper()
    plot_vsbc(r["favi_vsbc"], r["elbo_vsbc"], task_name)

In [ ]:
for vsbc_nc in vsbc_d_not_less_than_cases:
    plot_vsbc_for_case(result_dir / vsbc_nc)